In [1]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

Found existing installation: langchain 0.1.17
Uninstalling langchain-0.1.17:
  Successfully uninstalled langchain-0.1.17
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
  Using cached langchain-0.1.17-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
Using cached langchain-0.1.17-py3-none-any.whl (867 kB)
Using cached langchain_community-0.0.38-py3-none-any.whl (2.0 MB)
Using cached langchain_core-0.1.53-py3-none-any.whl (303 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Artificial_intelligence_in_India.pdf to Artificial_intelligence_in_India.pdf
Saving Climate_change_in_India.pdf to Climate_change_in_India.pdf
Saving COVID-19_pandemic.pdf to COVID-19_pandemic.pdf
Saving ISRO.pdf to ISRO.pdf
Saving Temples_of_modern_India.pdf to Temples_of_modern_India.pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

/usr/local/lib/python3.12/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


✅ Documents loaded: 232


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

✅ Total chunks: 979


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings ready


In [6]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

db.save_local("faiss_index")

print("✅ Database created")

✅ Database created


In [7]:
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

retriever = db.as_retriever(search_kwargs={"k": 3})

pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ Chatbot ready")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Chatbot ready


In [10]:
while True:
    query = input("Ask: ")

    result = qa(query)

    print("\nAnswer:", result["result"])

    print("\nSources:")
    for doc in result["source_documents"]:
        print(doc.metadata)

Ask: What is COVID impact?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

COVID-19  (https://www .cdc.gov/coronavirus/2019-ncov/index.html)  (Q&A (https://www .cdc.g
ov/coronavirus/2019-ncov/faq.html) ) by the US Centers for Disease Control and PreventionExternal links
Health agencies

By 2025, five years after the start of the pandemic, experts asked by The Independent  and The
Washington Post generally considered COVID-19 to have become endemic.[13][14] Cases and deaths from
COVID-19 still remained high; in the United States, the only two infectious diseases causing more than
10,000 deaths a year were flu and COVID, according to an expert in The W ashington Post .[13]
Despite strong economic rebounds following the initial lockdowns in early 2020, towards the latter
phases of the pandemic, many countries began to experience long-term economic effects. Several
countries saw high inflation

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


IndexError: index out of range in self